In [129]:
import sys
import os
sys.path.append('/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/')

import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats 
from scipy.stats import shapiro , kstest, mannwhitneyu, ttest_rel, wilcoxon
import torch
from torch import nn
from datasets.datasets import MRIDataset, STAREDataset

from helper_func.analyis_helper import clean_df, plot_boxplots, visualize, generate_batch_views, visualize_views, visualize_class_centroids, visualize_feats, prod_feats
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from skimage.measure import label
from monai.metrics import HausdorffDistanceMetric, ConfusionMatrixMetric, MeanIoU, DiceMetric

In [130]:
prediction_path = "/scratch-second/TTA_results/val_ent_only_decoder_lr6/logs/BraTS_GLI/BraTS-SSA-00057-000.npy"
prediction = np.load(prediction_path)

In [131]:
label_path = "/scratch_net/ken/radjoe/BraTS_SSA/Validation/labels_UNN/BraTS-SSA-00057-000-lbl.npy"
slabel = np.load(label_path)

In [132]:
prediction.shape

(4, 192, 224, 160)

In [133]:
hd_metric1 = HausdorffDistanceMetric(
    include_background=False,
    percentile=95.0,
    reduction="none"
    )

hd_metric2 = HausdorffDistanceMetric(
    include_background=False,
    percentile=95.0,
    reduction="none"
    )


dice_metric1 = DiceMetric(
    include_background=False,
    reduction="none",          
    ignore_empty=True
    )

dice_metric2 = DiceMetric(
    include_background=False,
    reduction="none",          
    ignore_empty=True
    )

In [ ]:

C, H, W, D = prediction.shape
# min_voxels = 100  # adjust for expected object size
filtered_vol = np.zeros_like(prediction)

for c in range(C):
    mask = prediction[c].astype(np.uint8)  # binary mask for class c
    min_voxels = mask.sum() * 0.1
    
    # 3D connected component labeling
    labeled_mask = label(mask, connectivity=3)
    
    # Filter small components
    filtered_mask = np.zeros_like(mask)
    for i in range(1, labeled_mask.max() + 1):
        if np.sum(labeled_mask == i) >= min_voxels:
            filtered_mask[labeled_mask == i] = 1
    
    filtered_vol[c] = filtered_mask

In [135]:
filtered_vol = torch.tensor(filtered_vol).unsqueeze(0).float() 
filtered_vol.shape

torch.Size([1, 4, 192, 224, 160])

In [136]:
prediction = torch.tensor(prediction).unsqueeze(0).float() 
prediction.shape
slabel = torch.tensor(slabel).unsqueeze(0).float() 
slabel.shape

torch.Size([1, 4, 192, 224, 160])

In [137]:
dice_metric1(prediction, slabel)

tensor([[0.7630, 0.7635, 0.5865]])

In [138]:
hd_metric1(prediction, slabel) 

tensor([[64.9199, 52.0961, 10.6771]])

In [139]:
hd_metric2(filtered_vol, slabel) 

tensor([[5.0990, 8.2037, 9.4868]])

In [140]:
dice_metric2(filtered_vol, slabel)

tensor([[0.8162, 0.8005, 0.5883]])